# Bài 2 — Titanic, Câu 7

Yêu cầu lập trình Titanic trang PDF 15–16

**Sinh viên:** NGUYỄN LỘC — **MSSV:** 6451071043

**Lưu ý:** Notebook tự chạy độc lập sau khi có dataset. Khi dùng Colab, cài `mlxtend`, `openpyxl` và các thư viện nếu cần.

In [ ]:
!pip -q install mlxtend openpyxl seaborn scikit-learn beautifulsoup4

In [ ]:
from pathlib import Path
import os, re, io, zipfile, time, warnings, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from sklearn.preprocessing import MinMaxScaler
from bs4 import BeautifulSoup
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')
HO_TEN = 'NGUYỄN LỘC'
MSSV = '6451071043'
BASE = Path('/content/lab1_data') if Path('/content').exists() else Path('/mnt/data')
BASE.mkdir(parents=True, exist_ok=True)
OUTPUT = BASE / 'results'; OUTPUT.mkdir(exist_ok=True)
def identity(): print(f'Họ tên: {HO_TEN} | MSSV: {MSSV}')
def pick_file(keywords, extensions=('.csv','.xlsx','.xls','.txt','.html','.htm')):
    fs=[p for p in BASE.rglob('*') if p.is_file() and p.suffix.lower() in extensions and not OUTPUT in p.parents]
    for keys in keywords:
        matches=[f for f in fs if all(w in f.name.lower() for w in keys)]
        if matches:return sorted(matches,key=lambda f:len(f.name))[0]
    raise FileNotFoundError(f'Không tìm thấy tập tin chứa từ khóa {keywords} trong {BASE}. File hiện có: {[x.name for x in fs[:30]]}')
def read_table(file, **kw):
    if file.suffix.lower() in ('.xls','.xlsx'):return pd.read_excel(file,**kw)
    for enc in ['utf-8-sig','utf-8','cp1252','latin-1']:
        try:
            df=pd.read_csv(file,encoding=enc,**kw)
            if df.shape[1]==1 and ';' in str(df.columns[0]):df=pd.read_csv(file,encoding=enc,sep=';',**kw)
            return df
        except UnicodeDecodeError:continue
    return pd.read_csv(file,encoding='latin-1',**kw)
def norm_cols(df):
    df=df.copy(); df.columns=[re.sub(r'[^a-z0-9]+','',str(c).strip().lower()) for c in df.columns]
    return df
def rule_table(freq, confidence):
    if freq.empty:return pd.DataFrame()
    try:r=association_rules(freq,metric='confidence',min_threshold=confidence,num_itemsets=int(freq.attrs.get('n_transactions',0)) or None)
    except TypeError:r=association_rules(freq,metric='confidence',min_threshold=confidence)
    if len(r):
        r=r.copy()
        for c in ['antecedents','consequents']:r[c]=r[c].map(lambda s:', '.join(sorted(map(str,s))))
    return r
identity(); print('Thư mục dữ liệu:', BASE)
# Tự dò dữ liệu được Colab upload vào /content hoặc LAB1_data.
def ensure_dataset(filename):
    for d in [BASE,Path('/content'),Path('/content/LAB1_data'),Path('/mnt/data')]:
        p=d/filename
        if p.exists(): return p
    try:
        from google.colab import files
        print('Upload dataset:',filename)
        upl=files.upload()
        for name,b in upl.items(): (BASE/Path(name).name).write_bytes(b)
        if (BASE/filename).exists():return BASE/filename
    except ImportError:pass
    raise FileNotFoundError(filename)


### Chuẩn bị — Đọc Titanic

In [ ]:
titanic_file=ensure_dataset('titanic.csv')
titanic=norm_cols(read_table(titanic_file));titanic=titanic.rename(columns={'survival':'survived','class':'pclass'})

### Chuẩn bị — Thuộc tính relatives

In [ ]:
identity()
t=titanic.copy()
for c in ['sibsp','parch']:
    if c not in t:t[c]=0
    t[c]=pd.to_numeric(t[c],errors='coerce').fillna(0)
t['relatives']=t['sibsp']+t['parch'];t['family_size']=t['relatives']+1
print('Đi một mình:',int((t.relatives==0).sum()),'| Có người thân:',int((t.relatives>0).sum()))
display(t.groupby('relatives')['survived'].agg(['count','mean']).head(15))
plt.figure(figsize=(9,4));sns.barplot(data=t,x='relatives',y='survived',errorbar=None);plt.title('Người thân đi cùng và tỷ lệ sống sót');plt.show()

### Chuẩn bị — Tiền xử lý

In [ ]:
identity()
t_clean=t.copy()
if 'name' in t_clean:t_clean['title']=t_clean['name'].astype(str).str.extract(r',\s*([^.]*)\.',expand=False).fillna('Unknown')
for c in ['age','fare']:
    if c in t_clean:
        t_clean[c]=pd.to_numeric(t_clean[c],errors='coerce')
        t_clean[c]=t_clean[c].fillna(t_clean[c].median())
for c in t_clean.select_dtypes(include=['object','string']).columns:
    t_clean[c]=t_clean[c].fillna(t_clean[c].mode(dropna=True).iloc[0] if t_clean[c].notna().any() else 'Unknown')
drop_cols=[c for c in ['ticket','cabin','boat','body','home.dest'] if c in t_clean]
t_clean=t_clean.drop(columns=drop_cols)
print('Đã bỏ các cột:',drop_cols);display(t_clean.isna().sum()[lambda x:x>0]);display(t_clean.head())

### Kết quả bài làm

In [14]:
identity()
if 'fare' in t_clean:t_clean['fare_int']=t_clean['fare'].round().astype('int64')
if 'sex' in t_clean:t_clean['sex_encoded']=t_clean['sex'].astype(str).str.lower().map({'male':0,'female':1}).fillna(-1).astype(int)
if 'embarked' in t_clean:
    t_clean['embarked_code']=pd.Categorical(t_clean['embarked']).codes
    embarked_oh=pd.get_dummies(t_clean['embarked'],prefix='embarked',dtype=int)
    t_clean=pd.concat([t_clean,embarked_oh],axis=1)
    print('Lưu ý: One-hot tránh tạo quan hệ thứ bậc giả giữa C, Q, S.')
# Nhóm đúng các khoảng gợi ý trong đề; không khẳng định mỗi nhóm sẽ bằng số lượng nhau.
if 'age' in t_clean:
    t_clean['age_group']=pd.cut(t_clean['age'],bins=[-np.inf,11,18,22,27,33,40,66,np.inf],labels=range(1,9),include_lowest=True)
    t_clean['age_group_qcut']=pd.qcut(t_clean['age'],q=8,duplicates='drop')
if 'fare' in t_clean:
    t_clean['fare_group']=pd.cut(t_clean['fare'],bins=[-np.inf,7.91,14.454,31,99,250,np.inf],labels=range(1,7))
    t_clean['fare_group_qcut']=pd.qcut(t_clean['fare'],q=6,duplicates='drop')
print('Thống kê nhóm tuổi:');display(t_clean['age_group'].value_counts().sort_index())
print('Thống kê nhóm giá vé:');display(t_clean['fare_group'].value_counts().sort_index())
display(t_clean.head())

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Họ tên:  NGUYỄN LỘC | MSSV: 6451071043
Lưu ý: One-hot tránh tạo quan hệ thứ bậc giả giữa C, Q, S.
Thống kê nhóm tuổi:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
age_group,
1,90
2,103
3,140
4,171
5,442
6,136
7,218
8,9


Thống kê nhóm giá vé:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,count
fare_group,
1,337
2,310
3,332
4,246
5,67
6,17


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,pclass,survived,name,sex,age,sibsp,parch,fare,embarked,homedest,...,fare_int,sex_encoded,embarked_code,embarked_C,embarked_Q,embarked_S,age_group,age_group_qcut,fare_group,fare_group_qcut
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,211.3375,S,"St Louis, MO",...,211,1,2,0,0,1,5,"(28.0, 29.0]",5,"(53.1, 512.329]"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,151.5500,S,"Montreal, PQ / Chesterville, ON",...,152,0,2,0,0,1,1,"(0.166, 18.0]",5,"(53.1, 512.329]"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,151.5500,S,"Montreal, PQ / Chesterville, ON",...,152,1,2,0,0,1,1,"(0.166, 18.0]",5,"(53.1, 512.329]"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,151.5500,S,"Montreal, PQ / Chesterville, ON",...,152,0,2,0,0,1,5,"(29.0, 35.0]",5,"(53.1, 512.329]"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,151.5500,S,"Montreal, PQ / Chesterville, ON",...,152,1,2,0,0,1,4,"(22.0, 27.0]",5,"(53.1, 512.329]"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag